# 22 — Held-out READ Go/No-Go decision

This notebook is model-free. It computes the preregistered held-out
concept-level AUCs and the single GO/NO-GO decision, then stops.

In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import matplotlib.pyplot as plt

from src.read_validation import (
    CANDIDATE_NAMES,
    READ_VALIDATION_PROTOCOL,
    READ_VALIDATION_PROTOCOL_SHA256,
    build_model_free_validation_report,
    read_json,
    write_json,
)

ROOT = Path.cwd()
RAW_DIR = ROOT / 'data/raw/v5'
implementation_sha256 = hashlib.sha256((ROOT / 'src/read_validation.py').read_bytes()).hexdigest()
raw20_path = RAW_DIR / '20_read_ground_truth.json'
raw21_path = RAW_DIR / '21_read_estimators.json'
raw20_sha256 = hashlib.sha256(raw20_path.read_bytes()).hexdigest()
raw21_sha256 = hashlib.sha256(raw21_path.read_bytes()).hexdigest()
metrics_before_decision = read_json(ROOT / 'results/metrics.json')['read_validation_v5']
assert metrics_before_decision['stage20']['raw_artifact']['sha256'] == raw20_sha256
assert metrics_before_decision['stage21']['raw_artifact']['sha256'] == raw21_sha256
raw20 = read_json(raw20_path)
raw21 = read_json(raw21_path)
assert raw20['protocol_sha256'] == raw21['protocol_sha256'] == READ_VALIDATION_PROTOCOL_SHA256
assert raw20['implementation_sha256'] == raw21['implementation_sha256'] == implementation_sha256
assert raw21['upstream_raw20_sha256'] == raw20_sha256
assert raw21['direction_bank_sha256'] == raw20['direction_bank_cache']['sha256']
assert raw21['candidate_names'] == list(CANDIDATE_NAMES)
scored_rows = raw21['rows']
assert len(scored_rows) == 163
identity_fields = ('concept_id', 'fold', 'label', 'class_name')
ground_identity = {row['row_id']: tuple(row[field] for field in identity_fields) for row in raw20['rows']}
score_identity = {row['row_id']: tuple(row[field] for field in identity_fields) for row in scored_rows}
assert ground_identity == score_identity and len(ground_identity) == 163
validation = build_model_free_validation_report(scored_rows)
decision = validation['go_no_go']
print(decision['one_line'])
print(json.dumps({candidate: validation['pooled_oof_auc']['candidate_results'][candidate] for candidate in CANDIDATE_NAMES}, indent=2))

READ NO-GO: no complete-coverage candidate reaches AUC >= 0.70 with the 95% bootstrap CI lower bound strictly above 0.50.
{
  "R1": {
    "candidate": "R1",
    "n_concepts": 79,
    "n_positive_concepts": 75,
    "n_negative_concepts": 4,
    "n_bootstrap": 10000,
    "bootstrap_seed": 1729,
    "bootstrap_unit": "dependency cluster: engine source/foil connected component or dashboard language",
    "bootstrap_stratification": "fixed provenance label",
    "n_dependency_clusters": 47,
    "cluster_counts_by_label": {
      "0": 4,
      "1": 43
    },
    "engine_pair_ids_retained_in_concept_metadata": true,
    "missing_concept_ids": [],
    "complete_concept_coverage": true,
    "status": "OK",
    "auc": 0.07833333333333332,
    "ci95_low": 0.02142857142857143,
    "ci95_high": 0.15203244109494102,
    "ci_method": "two-sided 95% percentile interval",
    "auc_threshold": 0.7,
    "ci_low_threshold_strictly_above": 0.5,
    "passes_go_threshold": false
  },
  "R2_sum": {
    "candi

In [2]:
fig_dir = ROOT / 'results/figures'
fig_dir.mkdir(parents=True, exist_ok=True)
auc_results = validation['pooled_oof_auc']['candidate_results']
x = np.arange(len(CANDIDATE_NAMES))
auc_values = [auc_results[name].get('auc', np.nan) if auc_results[name].get('auc') is not None else np.nan for name in CANDIDATE_NAMES]
low = [auc_results[name].get('ci95_low', np.nan) if auc_results[name].get('ci95_low') is not None else np.nan for name in CANDIDATE_NAMES]
high = [auc_results[name].get('ci95_high', np.nan) if auc_results[name].get('ci95_high') is not None else np.nan for name in CANDIDATE_NAMES]
yerr = np.asarray([
    [max(0.0, a-b) if np.isfinite(a) and np.isfinite(b) else 0.0 for a,b in zip(auc_values,low)],
    [max(0.0, b-a) if np.isfinite(a) and np.isfinite(b) else 0.0 for a,b in zip(auc_values,high)],
])

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(x, np.nan_to_num(auc_values, nan=0.0), color=['#6a1b9a'] + ['#00897b']*3 + ['#1565c0']*3)
finite = np.isfinite(auc_values)
if finite.any():
    ax.errorbar(x[finite], np.asarray(auc_values)[finite], yerr=yerr[:, finite], fmt='none', color='black', capsize=4)
for index, (bar, value) in enumerate(zip(bars, auc_values)):
    if np.isfinite(value):
        ax.text(bar.get_x()+bar.get_width()/2, value+0.025, f'{value:.3f}', ha='center', va='bottom', fontsize=9)
    else:
        ax.text(bar.get_x()+bar.get_width()/2, 0.03, 'NOT\nESTIMABLE', ha='center', va='bottom', fontsize=8, color='red')
ax.axhline(0.70, color='red', linestyle='--', label='GO point bar 0.70')
ax.axhline(0.50, color='black', linestyle=':', label='chance / CI-lower bar')
ax.set_xticks(x, CANDIDATE_NAMES, rotation=30, ha='right')
ax.set_ylim(0, 1.08)
ax.set_ylabel('Pooled out-of-fold ROC AUC')
ax.set_title('F1 — READ engine-vs-dashboard validation (79 semantic concepts)')
ax.legend()
fig.tight_layout()
f1 = fig_dir / 'f1_read_go_no_go_auc.png'
fig.savefig(f1, dpi=180)
plt.close(fig)

behavior_candidates = [name for name in CANDIDATE_NAMES if name.startswith(('R2_', 'R3_'))]
estimable = [name for name in behavior_candidates if auc_results[name]['status'] == 'OK']
best = max(estimable, key=lambda name: auc_results[name]['auc']) if estimable else None
concept_rows = validation['calibration']['concept_rows']
fig, ax = plt.subplots(figsize=(9, 7))
if best is None:
    best_auc = None
    best_ci = (None, None)
    ax.text(0.5, 0.5, 'NO BEHAVIOR-SPECIFIC READ IS ESTIMABLE', ha='center', va='center', transform=ax.transAxes, color='red')
    ax.set_axis_off()
    ax.set_title('F2 — behavior-specific READ vs coordinate-resampling A')
else:
    for class_name, color, marker in [('engine', '#1565c0', 'o'), ('dashboard', '#ef6c00', 's')]:
        subset = [row for row in concept_rows if row['class_name'] == class_name and row['A_causal_abs'] is not None and row['candidate_scores'][best] is not None]
        ax.scatter([row['A_causal_abs'] for row in subset], [row['candidate_scores'][best] for row in subset], c=color, marker=marker, alpha=0.8, label=f'{class_name} (N={len(subset)})')
    ax.set_xlabel('|CAUSAL(A)| coordinate-resampling effect')
    ax.set_ylabel(f'OOF train-CDF score: {best}')
    best_auc = auc_results[best].get('auc')
    best_ci = (auc_results[best].get('ci95_low'), auc_results[best].get('ci95_high'))
    ax.set_title(f'F2 — best behavior-specific READ vs coordinate-resampling A\n{best}: AUC={best_auc}, 95% CI={best_ci}')
    ax.legend()
fig.tight_layout()
f2 = fig_dir / 'f2_best_read_vs_patching.png'
fig.savefig(f2, dpi=180)
plt.close(fig)
print({'F1': str(f1), 'F2': str(f2), 'best_behavior_specific': best})

{'F1': '/home/jovyan/j-space-thoughts/results/figures/f1_read_go_no_go_auc.png', 'F2': '/home/jovyan/j-space-thoughts/results/figures/f2_best_read_vs_patching.png', 'best_behavior_specific': None}


In [3]:
raw22 = {
    'schema_version': 'read-go-no-go-decision-v1',
    'protocol_sha256': READ_VALIDATION_PROTOCOL_SHA256,
    'implementation_sha256': implementation_sha256,
    'upstream_raw20_sha256': raw20_sha256,
    'upstream_raw21_sha256': raw21_sha256,
    'validation': validation,
    'figures': [str(f1.relative_to(ROOT)), str(f2.relative_to(ROOT))],
}
raw22_artifact = write_json(RAW_DIR / '22_read_decision.json', raw22)

metrics_path = ROOT / 'results/metrics.json'
metrics = read_json(metrics_path)
v5 = metrics['read_validation_v5']
assert v5['protocol_sha256'] == READ_VALIDATION_PROTOCOL_SHA256
v5['stage22'] = {
    'status': 'COMPLETE',
    'raw_artifact': raw22_artifact,
    'upstream_raw20_sha256': raw20_sha256,
    'upstream_raw21_sha256': raw21_sha256,
    'figures': raw22['figures'],
    'validation': validation,
}
v5['status'] = 'COMPLETE'
v5['decision'] = decision['decision']
v5['decision_one_line'] = decision['one_line']
v5['hypothesis_status'] = 'NOT_TESTED'
v5['per_concept'] = validation['calibration']['concept_rows']
v5['auc_ci_table'] = validation['pooled_oof_auc']
write_json(metrics_path, metrics)

auc_lines = []
for name in CANDIDATE_NAMES:
    result = auc_results[name]
    auc_lines.append(
        f"| {name} | {result.get('status')} | {result.get('auc')} | "
        f"[{result.get('ci95_low')}, {result.get('ci95_high')}] | "
        f"{result.get('complete_concept_coverage')} | {result.get('passes_go_threshold')} |"
    )
correlation = validation['secondary_correlations']['A_B_correlation']
labels = validation['label_verification']
counts = validation['calibration']
stage20_counts = v5['stage20']['counts']
case_correlation = v5['stage20']['ground_truth_correlation_case_level']
label_failure_rows = [row for row in labels['rows'] if row['status'] == 'FAIL']
label_failure_reasons = {}
for row in label_failure_rows:
    for reason in row['failure_reasons']:
        label_failure_reasons[reason] = label_failure_reasons.get(reason, 0) + 1
report = f'''# READ Go/No-Go validation

## DECISION

**{decision['one_line']}**

This is a READ-method validation result only. Written-vs-Read P1/P2/P3 remain **NOT TESTED**.

## Preflight and model scope

- GPU: {v5['preflight']['gpu']['name']}; {v5['preflight']['gpu']['memory_total_mib']} MiB total, {v5['preflight']['gpu']['memory_free_mib']} MiB free.
- HF filesystem free: {v5['preflight']['disk']['free_gib']} GiB.
- Model: `Qwen/Qwen2.5-7B-Instruct`, bf16.
- Qwen3 scale arm: **SKIPPED** because no comparable validated Qwen3 J-Lens/direction instrument exists; 32B/30B also exceed disk and 4-bit cannot run the derivative protocol.

## Ground truth

Primary A is concept-coordinate resampling from a frozen clean donor. It is independent of READ weights/gradients but depends on the frozen J-Lens direction; it is not full-residual patching. Secondary B is the fixed masked source-to-foil clamped swap at alpha=1.5; it is not a source-only deletion.

- A defined: {stage20_counts['A_defined']}/163; B defined: {stage20_counts['B_defined']}/163. Undefined values were retained as missing.
- A/B finite paired case-level correlation: Spearman rho={case_correlation['spearman_rho']}; Pearson r={case_correlation['pearson_r']} (N={case_correlation['n_cases']}).
- Strict all-79-concept A/B correlation status: `{correlation['spearman']['status']}` because A or B is missing for {len(correlation['spearman']['missing_concept_ids'])} concepts; no pairwise-deleted concept statistic was substituted.

Declared-label verification: **{labels['status']}**; failures={labels['n_failures']}/163, all engines. Failure-reason counts={label_failure_reasons} (reasons overlap). Failures were retained. Clean capability was diagnostic only.

## Held-out classification

Inference uses {counts['n_concepts']} semantic concepts ({counts['concept_counts_by_class']['engine']} engines, {counts['concept_counts_by_class']['dashboard']} dashboard languages), not 163 independent concepts. Four grouped folds keep source/foil connected components and each language together. R2/R3 paths and R2 position profiles use training folds only; OOF scores use unlabeled training-concept CDF calibration.

| candidate | status | OOF AUC | cluster-bootstrap 95% CI | complete coverage | GO bar |
| --- | --- | ---: | --- | --- | --- |
{chr(10).join(auc_lines)}

The requested A-vs-B AUC rows are numerically identical because the fixed engine/dashboard labels—not causal magnitudes—define ROC AUC. They are one inferential look, not two chances to pass. Spearman associations against continuous A and B are retained in `metrics.json`.

R1 is complete but strongly ranks in the wrong direction (AUC={auc_results['R1']['auc']}; a post-hoc sign flip was not permitted). Every fold still selected a non-empty top-8 R2/R3 path and all 163 R3 derivative profiles completed. However, each training split contained 5–8 structurally unavailable A localizations. Under the preregistered complete-training-coverage rule, all six R2/R3 summary scores are non-estimable for the decision; none was converted to zero or another favorable value.

![F1](figures/f1_read_go_no_go_auc.png)

![F2](figures/f2_best_read_vs_patching.png)

## Estimators

- R1: global repaired-weight READ across every MLP/head in the compute-bounded common L25–27 frontier.
- R2: fixed train-fold top-8 path (2 MLP + 6 heads), with static sum and train-derived relative-position peak/carrying summaries.
- R3: exact input-dependent real-activation derivative on that same train-fold path, with position-normalized sum, peak, and carrying mean.

No estimator beyond R1/R2/R3 was added; no alpha sweep, controls, nulls, capability run, ambiguity, scale science, P1/P2/P3, tests, or lint were run.

## Limitations

- Only four independent dashboard concepts exist; the 10,000-draw CI resamples dependency clusters and is intrinsically fragile.
- Engine and dashboard labels are task/domain-confounded (two-hop facts versus multilingual continuation).
- The roster is reused calibration data, not an untouched external holdout.
- A uses direction-coordinate rather than full-state resampling because the repository has no matched clean/corrupt prompt pairs.
- Frozen donor activations are an exogenous corruption bank and may cross evaluation folds; donor folds are never used as labels or path-selection observations, but dependence remains.
- L25–27 is a compute-bounded common frontier, not all downstream components.

## Reproducibility

- Protocol SHA-256: `{READ_VALIDATION_PROTOCOL_SHA256}`.
- Notebook-20 raw: `{v5['stage20']['raw_artifact']['path']}` (`{v5['stage20']['raw_artifact']['sha256']}`).
- Notebook-21 raw: `{v5['stage21']['raw_artifact']['path']}` (`{v5['stage21']['raw_artifact']['sha256']}`).
- Notebook-22 raw: `{raw22_artifact['path']}` (`{raw22_artifact['sha256']}`).
'''
(ROOT / 'results/RESULTS.md').write_text(report, encoding='utf-8')
print(report)
assert decision['decision'] in {'GO', 'NO-GO'}
assert 'P1/P2/P3 remain **NOT TESTED**' in report
assert len(CANDIDATE_NAMES) == 7
print(decision['one_line'])

# READ Go/No-Go validation

## DECISION

**READ NO-GO: no complete-coverage candidate reaches AUC >= 0.70 with the 95% bootstrap CI lower bound strictly above 0.50.**

This is a READ-method validation result only. Written-vs-Read P1/P2/P3 remain **NOT TESTED**.

## Preflight and model scope

- GPU: NVIDIA H200; 143771 MiB total, 143072 MiB free.
- HF filesystem free: 38 GiB.
- Model: `Qwen/Qwen2.5-7B-Instruct`, bf16.
- Qwen3 scale arm: **SKIPPED** because no comparable validated Qwen3 J-Lens/direction instrument exists; 32B/30B also exceed disk and 4-bit cannot run the derivative protocol.

## Ground truth

Primary A is concept-coordinate resampling from a frozen clean donor. It is independent of READ weights/gradients but depends on the frozen J-Lens direction; it is not full-residual patching. Secondary B is the fixed masked source-to-foil clamped swap at alpha=1.5; it is not a source-only deletion.

- A defined: 155/163; B defined: 161/163. Undefined values were retained as missing.